# Unified Clinical Triage Multi-Agent Inference (`LangGraph` + Pre-Trained Models)
Loads your pre-trained models directly for inference:
* `healthcare_unsupervised_phenotyper.pkl` (`SklearnUnsupervisedPhenotyper`)
* `lightgbm_model.txt` (`LightGBMReadmissionExplainer`)
* `vitals_bilstm_attention.pth` (`PyTorchVitalsTrainer` / `VitalSignsBiLSTMAttentionNet`)
* `pneumonia_resnet18_gradcam.pth` (`TorchvisionChestXRayGradCAM`)
* `./biomedical_ner_model` (`HuggingFaceClinicalNLP`) & `ProductionClinicalHybridRAG`

In [ ]:
# !pip install -q datasets huggingface_hub medmnist scikit-learn lightgbm shap torch torchvision transformers sentence-transformers chromadb rank-bm25 langgraph langchain-core joblib matplotlib pandas numpy

In [1]:
from __future__ import annotations
import json
import re
import urllib.request
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, TypedDict

import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
import shap
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
import chromadb
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from medmnist import PneumoniaMNIST
from rank_bm25 import BM25Okapi
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from transformers import pipeline as hf_pipeline
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

/Users/yash/Library/A-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1105 12:05:30.666000 53948 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## 1. Model Class Definitions (Required for Unpickling & Weight Restoration)

In [2]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

In [3]:
class SklearnUnsupervisedPhenotyper:
    """Production Unsupervised ML pipeline using scikit-learn."""

    def __init__(self, n_pca_components: int = 5, n_clusters: int = 3, contamination: float = 0.03):
        self.imputer = KNNImputer(n_neighbors=5)
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=n_pca_components, random_state=42)
        self.kmeans = KMeans(n_clusters=n_clusters, init="k-means++", n_init=10, random_state=42)
        self.iforest = IsolationForest(n_estimators=150, contamination=contamination, random_state=42, n_jobs=-1)
        self.feature_names_: List[str] = []
        self.cluster_labels_map_: Dict[int, str] = {}

    def analyze_patient(self, patient_row: pd.DataFrame) -> Tuple[np.ndarray, Dict[str, Any]]:
        X_imp = self.imputer.transform(patient_row[self.feature_names_])
        X_scaled = self.scaler.transform(X_imp)
        X_pca = self.pca.transform(X_scaled)
        cid = int(self.kmeans.predict(X_pca)[0])
        anom_pred = int(self.iforest.predict(X_scaled)[0])
        anom_score = float(-self.iforest.score_samples(X_scaled)[0])

        return X_imp, {
            "cluster_id": cid,
            "phenotype_name": self.cluster_labels_map_.get(cid, f"Cluster {cid}"),
            "pca_coordinates": [round(float(x), 3) for x in X_pca[0]],
            "isolation_forest_anomaly_score": round(anom_score, 4),
            "is_outlier_flagged": bool(anom_pred == -1),
        }


class LightGBMReadmissionExplainer:
    """Wraps the Optuna-tuned LightGBM Booster (lightgbm_model.txt) and TreeSHAP explainer for patient inference."""

    def __init__(self, model_path: str = "lightgbm_model.txt", feature_names: Optional[List[str]] = None):
        self.booster = lgb.Booster(model_file=model_path)
        self.explainer = shap.TreeExplainer(self.booster)
        self.feature_names_ = feature_names or []

    def predict_and_explain(self, x_imp_row: np.ndarray, feature_names: Optional[List[str]] = None) -> Dict[str, Any]:
        """Runs single-patient inference with the loaded LightGBM Booster and computes exact TreeSHAP drivers."""
        feat_cols = feature_names or self.feature_names_
        prob = float(self.booster.predict(x_imp_row)[0])
        shap_vals = self.explainer.shap_values(x_imp_row)
        if isinstance(shap_vals, list):
            sv_row = shap_vals[1][0]
        elif np.asarray(shap_vals).ndim == 3:
            sv_row = shap_vals[0, :, 1]
        else:
            sv_row = shap_vals[0]

        ranked = sorted(
            zip(feat_cols, x_imp_row[0], sv_row),
            key=lambda item: abs(float(item[2])),
            reverse=True,
        )
        top_drivers = [
            {
                "feature": fname,
                "value": round(float(val), 2),
                "shap_log_odds_impact": round(float(sv), 4),
                "direction": "increases_risk" if sv > 0 else "decreases_risk",
            }
            for fname, val, sv in ranked[:5]
        ]
        return {
            "readmission_or_icu_risk_prob": round(prob, 4),
            "risk_tier": "HIGH" if prob >= 0.40 else ("MODERATE" if prob >= 0.25 else "LOW"),
            "top_treeshap_drivers": top_drivers,
        }


class VitalSignsBiLSTMAttentionNet(nn.Module):
    def __init__(self, in_channels: int = 5, hidden_dim: int = 32, num_heads: int = 4):
        super().__init__()
        self.conv1d = nn.Sequential(
            nn.Conv1d(in_channels, hidden_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
        )
        self.bilstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.2,
        )
        self.attn = nn.MultiheadAttention(embed_dim=hidden_dim * 2, num_heads=num_heads, batch_first=True)
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim * 2),
            nn.Linear(hidden_dim * 2, 32),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x_conv = self.conv1d(x.transpose(1, 2)).transpose(1, 2)
        lstm_out, _ = self.bilstm(x_conv)
        attn_out, attn_weights = self.attn(lstm_out, lstm_out, lstm_out)
        pooled = attn_out.mean(dim=1)
        logits = self.classifier(pooled).squeeze(-1)
        temporal_importance = attn_weights.mean(dim=1)
        return logits, temporal_importance


class PyTorchVitalsTrainer:
    def __init__(self, epochs: int = 25, lr: float = 2e-3):
        self.device = get_device()
        self.epochs = epochs
        self.lr = lr
        self.net = VitalSignsBiLSTMAttentionNet().to(self.device)
        self.mean_: Optional[np.ndarray] = None
        self.std_: Optional[np.ndarray] = None

    def forecast_trajectory(self, vitals_24h: np.ndarray) -> Dict[str, Any]:
        self.net.eval()
        x_norm = (vitals_24h[None, :, :] - self.mean_) / self.std_
        with torch.no_grad():
            tx = torch.tensor(x_norm, dtype=torch.float32, device=self.device)
            logit, attn_map = self.net(tx)
            prob = float(torch.sigmoid(logit).item())
            hour_weights = attn_map[0].cpu().numpy()

        peak_hours = [int(h) for h in np.argsort(hour_weights)[-3:][::-1]]
        slope = (vitals_24h[-1] - vitals_24h[-6]) / 5.0
        proj_6h = vitals_24h[-1] + 6.0 * 0.5 * slope
        return {
            "deterioration_6h_prob": round(prob, 4),
            "peak_attention_hours": peak_hours,
            "projected_vitals_6h": {
                "heart_rate_bpm": round(float(proj_6h[0]), 1),
                "spo2_pct": round(float(np.clip(proj_6h[1], 70.0, 100.0)), 1),
                "systolic_bp_mmhg": round(float(proj_6h[2]), 1),
                "resp_rate_bpm": round(float(proj_6h[3]), 1),
                "temp_c": round(float(proj_6h[4]), 1),
            },
        }


class TorchvisionChestXRayGradCAM:
    def __init__(self):
        self.device = get_device()
        self.model = tv_models.resnet18(weights=None)
        self.model.fc = nn.Linear(self.model.fc.in_features, 1)
        self.model = self.model.to(self.device)
        self._feature_maps: Optional[torch.Tensor] = None
        self._gradients: Optional[torch.Tensor] = None
        self._register_gradcam_hooks()

    def _register_gradcam_hooks(self) -> None:
        target_layer = self.model.layer4[-1]
        target_layer.register_forward_hook(lambda _m, _i, out: setattr(self, "_feature_maps", out))
        target_layer.register_full_backward_hook(lambda _m, _gi, go: setattr(self, "_gradients", go[0]))

    @staticmethod
    def _prepare_batch(imgs_uint8: np.ndarray) -> torch.Tensor:
        t = torch.tensor(imgs_uint8, dtype=torch.float32) / 255.0
        if t.ndim == 3:
            t = t.unsqueeze(1)
        elif t.ndim == 4 and t.shape[-1] == 1:
            t = t.permute(0, 3, 1, 2)
        t = F.interpolate(t, size=(64, 64), mode="bilinear", align_corners=False)
        return t.repeat(1, 3, 1, 1)

    def predict_with_gradcam(self, img_28x28: np.ndarray) -> Dict[str, Any]:
        self.model.eval()
        x = self._prepare_batch(img_28x28[None, ...]).to(self.device)
        x.requires_grad_(True)
        self.model.zero_grad()
        logit = self.model(x).squeeze()
        prob = float(torch.sigmoid(logit).item())
        logit.backward()

        grads = self._gradients[0]
        fmaps = self._feature_maps[0]
        weights = grads.mean(dim=(1, 2))
        cam = torch.relu(torch.sum(weights[:, None, None] * fmaps, dim=0))
        cam_up = F.interpolate(cam[None, None, ...], size=(28, 28), mode="bilinear", align_corners=False)[0, 0]
        cam_np = cam_up.detach().cpu().numpy()
        if cam_np.max() > 1e-8:
            cam_np /= cam_np.max()

        r, c = np.unravel_index(np.argmax(cam_np), cam_np.shape)
        side = "Right" if c >= 14 else "Left"
        zone = "Lower Lobe" if r >= 14 else "Upper/Mid Lobe"

        return {
            "pneumonia_probability": round(prob, 4),
            "radiology_impression": f"Pulmonary consolidation localized to {side} {zone}" if prob >= 0.5 else "Clear lung fields without focal opacity",
            "gradcam_hotspot_rc": [int(r), int(c)],
            "gradcam_anatomical_zone": f"{side} {zone}",
        }


class HuggingFaceClinicalNLP:
    NEGATION_PATTERNS = ("denies", "denied", "no evidence of", "without", "negative for", "unremarkable for")

    def __init__(self, model_name: str = "./biomedical_ner_model"):
        if torch.cuda.is_available():
            device_id = 0       # NVIDIA GPU (Colab)
        elif torch.backends.mps.is_available():
            device_id = "mps"   # Apple Silicon GPU
        else:
            device_id = -1      # CPU
        target_model = model_name if Path(model_name).exists() else "d4data/biomedical-ner-all"
        print(f"Loading Biomedical NER model from: {target_model}...")
        self.ner_pipe = hf_pipeline("ner", model=target_model, aggregation_strategy="simple", device=device_id)

    def extract_clinical_entities(self, note_text: str) -> Dict[str, Any]:
        raw_entities = self.ner_pipe(note_text)
        affirmed_problems, negated_problems, medications, diagnostic_procedures = [], [], [], []
        note_lower = note_text.lower()
        for ent in raw_entities:
            word = ent["word"].replace("##", "").strip()
            if len(word) < 3:
                continue
            label = ent["entity_group"].upper()
            start = int(ent.get("start", 0))
            context_window = note_lower[max(0, start - 45) : start]
            is_negated = any(pat in context_window for pat in self.NEGATION_PATTERNS)

            if label in {"SIGN_SYMPTOM", "DISEASE_DISORDER"}:
                (negated_problems if is_negated else affirmed_problems).append(word)
            elif label in {"MEDICATION", "THERAPEUTIC_PROCEDURE"}:
                medications.append(word)
            elif label in {"DIAGNOSTIC_PROCEDURE", "LAB_VALUE"}:
                diagnostic_procedures.append(word)

        allergies = [
            alg.capitalize()
            for alg in ("penicillin", "sulfa", "cephalosporin", "aspirin", "vancomycin")
            if alg in note_lower and ("allerg" in note_lower or "anaphylaxis" in note_lower)
        ]
        return {
            "affirmed_clinical_findings": sorted(list(set(affirmed_problems))),
            "negated_clinical_findings": sorted(list(set(negated_problems))),
            "extracted_medications": sorted(list(set(medications))),
            "diagnostic_procedures": sorted(list(set(diagnostic_procedures))),
            "documented_allergies": allergies if allergies else ["NKDA"],
        }


class ProductionClinicalHybridRAG:
    def __init__(
        self,
        corpus: List[Dict[str, str]],
        embed_model_name: str = "NeuML/pubmedbert-base-embeddings",
        reranker_model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2",
    ):
        self.corpus = corpus
        self.embedder = SentenceTransformer(embed_model_name)
        self.reranker = CrossEncoder(reranker_model_name)
        self.chroma_client = chromadb.Client()
        self.collection = self.chroma_client.get_or_create_collection(
            name="pubmedqa_clinical_guidelines",
            metadata={"hnsw:space": "cosine"},
        )
        texts = [f"{doc['title']}. {doc['text']}" for doc in self.corpus]
        ids = [doc["doc_id"] for doc in self.corpus]
        embeddings = self.embedder.encode(texts, show_progress_bar=False, normalize_embeddings=True).tolist()
        self.collection.upsert(
            ids=ids,
            documents=texts,
            embeddings=embeddings,
            metadatas=[{"title": d["title"], "source": d["source"]} for d in self.corpus],
        )
        tokenized_corpus = [re.findall(r"[a-z0-9]+", t.lower()) for t in texts]
        self.bm25 = BM25Okapi(tokenized_corpus)
        self.id_to_idx = {doc["doc_id"]: idx for idx, doc in enumerate(self.corpus)}

    def hybrid_search_with_rerank(self, query: str, top_k: int = 3) -> List[Dict[str, Any]]:
        q_emb = self.embedder.encode([query], normalize_embeddings=True).tolist()
        dense_ids = self.collection.query(query_embeddings=q_emb, n_results=min(10, len(self.corpus)))["ids"][0]
        q_tokens = re.findall(r"[a-z0-9]+", query.lower())
        bm25_top = np.argsort(-self.bm25.get_scores(q_tokens))[:10]
        bm25_ids = [self.corpus[int(i)]["doc_id"] for i in bm25_top]

        rrf_scores: Dict[str, float] = {}
        for rank, doc_id in enumerate(dense_ids):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (60.0 + rank + 1)
        for rank, doc_id in enumerate(bm25_ids):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (60.0 + rank + 1)

        candidate_ids = sorted(rrf_scores.keys(), key=lambda d: rrf_scores[d], reverse=True)[:8]
        candidates = [self.corpus[self.id_to_idx[cid]] for cid in candidate_ids]
        pairs = [[query, f"{c['title']}. {c['text']}"] for c in candidates]
        ce_scores = self.reranker.predict(pairs)
        ranked_order = np.argsort(-np.asarray(ce_scores))[:top_k]

        return [
            {
                "doc_id": candidates[int(idx)]["doc_id"],
                "title": candidates[int(idx)]["title"],
                "source": candidates[int(idx)]["source"],
                "snippet": candidates[int(idx)]["text"][:450] + "...",
                "cross_encoder_score": round(float(ce_scores[int(idx)]), 4),
            }
            for idx in ranked_order
        ]

## 2. Load All Pre-Trained Models (`pkl`, `txt`, `pth`, `biomedical_ner_model`)

In [4]:
# 1. Load Unsupervised Phenotyper
unsup_engine = joblib.load("healthcare_unsupervised_phenotyper.pkl")
feature_names = unsup_engine.feature_names_
print("Loaded healthcare_unsupervised_phenotyper.pkl | Features:", feature_names)

# 2. Load Optuna-Tuned LightGBM Model & TreeSHAP Explainer Class
lgbm_engine = LightGBMReadmissionExplainer(model_path="lightgbm_model.txt", feature_names=feature_names)
print("Loaded lightgbm_model.txt into LightGBMReadmissionExplainer")

# 3. Load Trained Bi-LSTM + Attention ICU Vitals Model
vitals_ckpt = torch.load("vitals_bilstm_attention.pth", map_location="cpu", weights_only=False)
vitals_trainer = PyTorchVitalsTrainer(epochs=vitals_ckpt["epochs"], lr=vitals_ckpt["lr"])
vitals_trainer.net.load_state_dict(vitals_ckpt["model_state_dict"])
vitals_trainer.mean_ = np.asarray(vitals_ckpt["mean"])
vitals_trainer.std_ = np.asarray(vitals_ckpt["std"])
vitals_trainer.net.eval()
print("Loaded vitals_bilstm_attention.pth")

# 4. Load Trained Chest X-Ray ResNet-18 Grad-CAM Model
xray_ckpt = torch.load("pneumonia_resnet18_gradcam.pth", map_location="cpu", weights_only=True)
xray_engine = TorchvisionChestXRayGradCAM()
xray_engine.model.load_state_dict(xray_ckpt["model_state_dict"])
xray_engine.model.eval()
print("Loaded pneumonia_resnet18_gradcam.pth")

# 5. Load Saved Biomedical NER Model & Hybrid RAG Corpus
nlp_engine = HuggingFaceClinicalNLP(model_name="./biomedical_ner_model")

medqa_ds = load_dataset("GBaker/MedQA-USMLE-4-options", split="train[:30]")
usmle_notes = [row["question"] for row in medqa_ds]
pubmed_ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train[:300]")
pubmed_corpus = []
for i, row in enumerate(pubmed_ds):
    ctx_list = row["context"]["contexts"] if isinstance(row["context"], dict) else [str(row["context"])]
    abstract_text = " ".join(ctx_list) + " Conclusion: " + str(row.get("long_answer", ""))
    pubmed_corpus.append({
        "doc_id": f"PUBMED-{row.get('pubid', 10000 + i)}",
        "title": str(row["question"]),
        "source": f"PubMed ID {row.get('pubid', i)} ({row.get('final_decision', 'clinical')})",
        "text": abstract_text[:1500],
    })
rag_engine = ProductionClinicalHybridRAG(corpus=pubmed_corpus)

Loaded healthcare_unsupervised_phenotyper.pkl | Features: ['age_years', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']
Loaded lightgbm_model.txt into LightGBMReadmissionExplainer
Loaded vitals_bilstm_attention.pth
Loaded pneumonia_resnet18_gradcam.pth
Loading Biomedical NER model from: ./biomedical_ner_model...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6421.26it/s]


## 3. Build `LangGraph` Clinical Triage Workflow with Human-in-the-Loop Checkpointing

In [5]:
class TriageAgentState(TypedDict):
    """Typed state dictionary flowing through the LangGraph clinical triage workflow."""
    patient_id: str
    tabular_row_json: str
    vitals_24h_list: List[List[float]]
    xray_28x28_list: List[List[int]]
    clinical_note: str
    unsupervised_phenotype_report: Dict[str, Any]
    tabular_risk_shap_report: Dict[str, Any]
    vitals_trajectory_report: Dict[str, Any]
    chest_xray_gradcam_report: Dict[str, Any]
    clinical_nlp_report: Dict[str, Any]
    hybrid_rag_citations: List[Dict[str, Any]]
    composite_acuity_score: float
    esi_triage_level: str
    safety_guardrail_alerts: List[str]
    recommended_clinical_orders: List[str]
    hitl_clinician_status: str
    execution_trace: List[str]


def build_langgraph_triage_workflow(
    unsup_model: SklearnUnsupervisedPhenotyper,
    tabular_model: LightGBMReadmissionExplainer,
    vitals_model: PyTorchVitalsTrainer,
    xray_model: TorchvisionChestXRayGradCAM,
    nlp_model: HuggingFaceClinicalNLP,
    rag_model: ProductionClinicalHybridRAG,
):
    @tool
    def analyze_patient_phenotype_and_readmission_risk(tabular_row_json: str) -> Dict[str, Any]:
        """Runs scikit-learn Phenotyper and LightGBM + TreeSHAP on patient tabular features."""
        row_df = pd.DataFrame([json.loads(tabular_row_json)])
        x_imp, unsup_rep = unsup_model.analyze_patient(row_df)
        risk_rep = tabular_model.predict_and_explain(x_imp, unsup_model.feature_names_)
        return {"unsupervised": unsup_rep, "tabular_shap": risk_rep}

    @tool
    def forecast_icu_vitals_trajectory(vitals_24h_list: List[List[float]]) -> Dict[str, Any]:
        """Runs PyTorch Bi-LSTM + MultiheadAttention over 24-hour ICU vital streams."""
        arr = np.array(vitals_24h_list, dtype=np.float32)
        return vitals_model.forecast_trajectory(arr)

    @tool
    def analyze_chest_xray_and_clinical_note(xray_28x28_list: List[List[int]], clinical_note: str) -> Dict[str, Any]:
        """Runs Torchvision ResNet-18 + Grad-CAM on chest X-ray and HuggingFace BioNER on clinical note."""
        img = np.array(xray_28x28_list, dtype=np.uint8)
        vis_rep = xray_model.predict_with_gradcam(img)
        nlp_rep = nlp_model.extract_clinical_entities(clinical_note)
        return {"vision_gradcam": vis_rep, "clinical_nlp": nlp_rep}

    @tool
    def retrieve_clinical_evidence_rag(clinical_query: str) -> List[Dict[str, Any]]:
        """Searches ChromaDB (PubMedBERT) + BM25Okapi and reranks biomedical evidence with CrossEncoder."""
        return rag_model.hybrid_search_with_rerank(clinical_query, top_k=3)

    def node_tabular_phenotyping_and_shap(state: TriageAgentState) -> Dict[str, Any]:
        res = analyze_patient_phenotype_and_readmission_risk.invoke({"tabular_row_json": state["tabular_row_json"]})
        trace = state.get("execution_trace", []) + [
            f"[Tabular & Phenotyping Node] Phenotype='{res['unsupervised']['phenotype_name']}' | Readmission Risk={res['tabular_shap']['readmission_or_icu_risk_prob']:.1%}"
        ]
        return {
            "unsupervised_phenotype_report": res["unsupervised"],
            "tabular_risk_shap_report": res["tabular_shap"],
            "execution_trace": trace,
        }

    def node_temporal_vitals_dl(state: TriageAgentState) -> Dict[str, Any]:
        res = forecast_icu_vitals_trajectory.invoke({"vitals_24h_list": state["vitals_24h_list"]})
        trace = state["execution_trace"] + [
            f"[Vitals Bi-LSTM Node] 6h Deterioration={res['deterioration_6h_prob']:.1%} | Peak Hours={res['peak_attention_hours']}"
        ]
        return {"vitals_trajectory_report": res, "execution_trace": trace}

    def node_multimodal_vision_and_nlp(state: TriageAgentState) -> Dict[str, Any]:
        res = analyze_chest_xray_and_clinical_note.invoke({
            "xray_28x28_list": state["xray_28x28_list"],
            "clinical_note": state["clinical_note"],
        })
        trace = state["execution_trace"] + [
            f"[Vision & BioNER Node] X-Ray Pneumonia={res['vision_gradcam']['pneumonia_probability']:.1%} | Allergies={res['clinical_nlp']['documented_allergies']}"
        ]
        return {
            "chest_xray_gradcam_report": res["vision_gradcam"],
            "clinical_nlp_report": res["clinical_nlp"],
            "execution_trace": trace,
        }

    def node_hybrid_rag_retrieval(state: TriageAgentState) -> Dict[str, Any]:
        nlp_rep = state["clinical_nlp_report"]
        vis_rep = state["chest_xray_gradcam_report"]
        findings = " ".join(nlp_rep["affirmed_clinical_findings"][:4])
        meds = " ".join(nlp_rep["extracted_medications"][:3])
        allergies = " ".join(nlp_rep["documented_allergies"])
        query = f"clinical management {vis_rep['radiology_impression']} {findings} {meds} allergy {allergies}".strip()

        citations = retrieve_clinical_evidence_rag.invoke({"clinical_query": query})
        trace = state["execution_trace"] + [
            f"[Hybrid RAG Node] Retrieved & CrossEncoder-reranked {len(citations)} PubMedQA documents"
        ]
        return {"hybrid_rag_citations": citations, "execution_trace": trace}

    def node_synthesis_and_safety_guardrails(state: TriageAgentState) -> Dict[str, Any]:
        p_tab = state["tabular_risk_shap_report"]["readmission_or_icu_risk_prob"]
        p_seq = state["vitals_trajectory_report"]["deterioration_6h_prob"]
        p_vis = state["chest_xray_gradcam_report"]["pneumonia_probability"]
        composite = float(0.40 * p_tab + 0.35 * p_seq + 0.25 * p_vis)

        esi = (
            "ESI Level 1 — Immediate ICU / Resuscitation Triage"
            if composite >= 0.60
            else ("ESI Level 2 — High-Risk Inpatient Step-Down" if composite >= 0.35 else "ESI Level 4 — Low-Acuity Observation")
        )

        alerts: List[str] = []
        orders: List[str] = []
        nlp_rep = state["clinical_nlp_report"]
        cited_ids = [c["doc_id"] for c in state["hybrid_rag_citations"]]

        if state["unsupervised_phenotype_report"]["is_outlier_flagged"]:
            alerts.append("ISOLATION FOREST OUTLIER: Extreme clinical/sensor profile detected; verify bedside vitals manually.")

        if "Penicillin" in nlp_rep["documented_allergies"] and p_vis >= 0.50:
            alerts.append("ALLERGY GUARDRAIL: Documented Penicillin allergy — Beta-Lactam/Ceftriaxone orders blocked.")
            orders.append(f"Administer respiratory fluoroquinolone (Levofloxacin 750mg IV) or Aztreonam + Vancomycin (Grounded in {cited_ids}).")
        elif p_vis >= 0.50:
            orders.append(f"Initiate empiric CAP antimicrobial regimen and obtain sputum/blood cultures (Grounded in {cited_ids}).")

        if composite >= 0.50:
            orders.append(f"Continuous telemetry + hourly SpO2/MAP trend checks; repeat STAT serum lactate (Grounded in {cited_ids}).")
        else:
            orders.append(f"Standard observation protocol and 48h outpatient follow-up (Grounded in {cited_ids}).")

        trace = state["execution_trace"] + [
            f"[Guardrail Synthesis Node] Composite Risk={composite:.1%} ({esi}) with {len(alerts)} Safety Alerts"
        ]
        return {
            "composite_acuity_score": round(composite, 4),
            "esi_triage_level": esi,
            "safety_guardrail_alerts": alerts,
            "recommended_clinical_orders": orders,
            "hitl_clinician_status": "PENDING_PHYSICIAN_APPROVAL" if (composite >= 0.35 or alerts) else "AUTO_CLEARED_LOW_RISK",
            "execution_trace": trace,
        }

    def node_clinician_hitl_signoff(state: TriageAgentState) -> Dict[str, Any]:
        trace = state["execution_trace"] + ["[Clinician HITL Node] Attending Physician Sign-Off Completed -> Orders Released"]
        return {"hitl_clinician_status": "APPROVED_BY_ATTENDING_PHYSICIAN", "execution_trace": trace}

    builder = StateGraph(TriageAgentState)
    builder.add_node("tabular_ml_node", node_tabular_phenotyping_and_shap)
    builder.add_node("temporal_dl_node", node_temporal_vitals_dl)
    builder.add_node("vision_nlp_node", node_multimodal_vision_and_nlp)
    builder.add_node("hybrid_rag_node", node_hybrid_rag_retrieval)
    builder.add_node("guardrail_synthesis_node", node_synthesis_and_safety_guardrails)
    builder.add_node("clinician_hitl_signoff", node_clinician_hitl_signoff)

    builder.add_edge(START, "tabular_ml_node")
    builder.add_edge("tabular_ml_node", "temporal_dl_node")
    builder.add_edge("temporal_dl_node", "vision_nlp_node")
    builder.add_edge("vision_nlp_node", "hybrid_rag_node")
    builder.add_edge("hybrid_rag_node", "guardrail_synthesis_node")
    builder.add_edge("guardrail_synthesis_node", "clinician_hitl_signoff")
    builder.add_edge("clinician_hitl_signoff", END)

    memory = MemorySaver()
    return builder.compile(checkpointer=memory, interrupt_before=["clinician_hitl_signoff"])


langgraph_app = build_langgraph_triage_workflow(
    unsup_model=unsup_engine,
    tabular_model=lgbm_engine,
    vitals_model=vitals_trainer,
    xray_model=xray_engine,
    nlp_model=nlp_engine,
    rag_model=rag_engine,
)

## 4. Load a Real Test Patient & Execute `LangGraph` Inference with HITL Pause & Resume

In [7]:
# Load sample test inputs (or pass any custom patient record)
csv_path = hf_hub_download(repo_id="Bena345/diabetes-readmission", filename="train.csv", repo_type="dataset")
df_sample = pd.read_csv(csv_path, nrows=100)
df_sample["age_years"] = df_sample["age"].astype(str).str.extract(r"(\d+)")[0].astype(float) + 5.0
sample_tabular_row = df_sample[unsup_engine.feature_names_].iloc[0]

pneumonia_test = PneumoniaMNIST(split="test", download=True, size=28)
pos_xray_idx = int(np.where(pneumonia_test.labels.squeeze() == 1)[0][0])

# Create a 24-hour vital stream tensor (or use X_seq[0] from MIMIC-IV Demo)
sample_vitals_24h = np.tile(np.array([96.0, 92.0, 104.0, 24.0, 38.2], dtype=np.float32), (24, 1))

sample_note = (
    usmle_notes[0]
    + " Patient has a documented severe allergy to Penicillin (anaphylaxis). Currently taking Warfarin for atrial fibrillation."
)

initial_state: TriageAgentState = {
    "patient_id": "MIMIC-REAL-1001",
    "tabular_row_json": sample_tabular_row.to_json(),
    "vitals_24h_list": sample_vitals_24h.tolist(),
    "xray_28x28_list": pneumonia_test.imgs[pos_xray_idx].squeeze().tolist(),
    "clinical_note": sample_note,
    "unsupervised_phenotype_report": {},
    "tabular_risk_shap_report": {},
    "vitals_trajectory_report": {},
    "chest_xray_gradcam_report": {},
    "clinical_nlp_report": {},
    "hybrid_rag_citations": [],
    "composite_acuity_score": 0.0,
    "esi_triage_level": "",
    "safety_guardrail_alerts": [],
    "recommended_clinical_orders": [],
    "hitl_clinician_status": "",
    "execution_trace": [],
}

thread_cfg = {"configurable": {"thread_id": "colab-clinical-triage-thread-1"}}

# Run graph until the Human-in-the-Loop interrupt point (`clinician_hitl_signoff`)
paused_state = langgraph_app.invoke(initial_state, config=thread_cfg)
print("\n" + "=" * 88)
print(f"[LangGraph HITL Pause] Patient: {paused_state['patient_id']} | Status: {paused_state['hitl_clinician_status']}")
print(f"Acuity Classification : {paused_state['esi_triage_level']} (Composite Risk: {paused_state['composite_acuity_score']:.1%})")
print(f"Patient Phenotype     : {paused_state['unsupervised_phenotype_report']['phenotype_name']}")
print(f"LightGBM + TreeSHAP   : Risk={paused_state['tabular_risk_shap_report']['readmission_or_icu_risk_prob']:.1%} | Top SHAP={paused_state['tabular_risk_shap_report']['top_treeshap_drivers'][:3]}")
print(f"PyTorch Bi-LSTM Vitals: 6h Deterioration={paused_state['vitals_trajectory_report']['deterioration_6h_prob']:.1%} | Peak Hours={paused_state['vitals_trajectory_report']['peak_attention_hours']}")
print(f"ResNet-18 Grad-CAM    : {paused_state['chest_xray_gradcam_report']['radiology_impression']} (p={paused_state['chest_xray_gradcam_report']['pneumonia_probability']:.1%})")
print(f"Biomedical NER        : Findings={paused_state['clinical_nlp_report']['affirmed_clinical_findings'][:5]} | Allergies={paused_state['clinical_nlp_report']['documented_allergies']}")
for alert in paused_state["safety_guardrail_alerts"]:
    print(f"  [!] SAFETY ALERT    : {alert}")
for order in paused_state["recommended_clinical_orders"]:
    print(f"  [+] PROPOSED ORDER  : {order}")

# Resume LangGraph execution after Attending Physician approves HITL checkpoint
final_state = langgraph_app.invoke(None, config=thread_cfg)
print(f"\n[LangGraph Resumed] Final HITL Status: {final_state['hitl_clinician_status']}")

/Users/yash/Library/A-project/.venv/lib/python3.12/site-packages/shap/explainers/_tree.py:578: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(



[LangGraph HITL Pause] Patient: MIMIC-REAL-1001 | Status: PENDING_PHYSICIAN_APPROVAL
Acuity Classification : ESI Level 1 — Immediate ICU / Resuscitation Triage (Composite Risk: 64.9%)
Patient Phenotype     : Phenotype 2: Moderate Chronic Multi-Medication Cohort
LightGBM + TreeSHAP   : Risk=18.9% | Top SHAP=[{'feature': 'number_inpatient', 'value': 0.0, 'shap_log_odds_impact': -0.18, 'direction': 'decreases_risk'}, {'feature': 'num_procedures', 'value': 3.0, 'shap_log_odds_impact': -0.0718, 'direction': 'decreases_risk'}, {'feature': 'time_in_hospital', 'value': 5.0, 'shap_log_odds_impact': 0.0616, 'direction': 'increases_risk'}]
PyTorch Bi-LSTM Vitals: 6h Deterioration=92.3% | Peak Hours=[18, 17, 19]
ResNet-18 Grad-CAM    : Pulmonary consolidation localized to Left Upper/Mid Lobe (p=100.0%)
Biomedical NER        : Findings=['costovertebral angle', 'ergy', 'feels well', 'tenderness'] | Allergies=['Penicillin']
  [!] SAFETY ALERT    : ALLERGY GUARDRAIL: Documented Penicillin allergy — B